# 🌿 Plant Disease Classification: train on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amr-elgharip2006/Plant-Disease-Classification/blob/main/notebooks/train_colab.ipynb)

Runs the full pipeline from [the repo](https://github.com/amr-elgharip2006/Plant-Disease-Classification): download → clean → split → train (head + fine-tune) → evaluate → demo.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
!nvidia-smi -L
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Get the code

In [ ]:
!git clone -q https://github.com/amr-elgharip2006/Plant-Disease-Classification.git
%cd Plant-Disease-Classification
!pip install -q kagglehub gradio

## 2. (Optional) keep results on Google Drive
Colab wipes its disk when the session ends. Run this to save `artifacts/` to Drive instead.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/plant-disease-artifacts"

In [ ]:
# Run this instead if you skipped Drive
ARTIFACTS = "artifacts"

## 3. Explore and clean the data
Downloads PlantVillage, removes corrupted and duplicate images, and creates a stratified 70/15/15 split.

In [ ]:
!python explore.py --output-dir "$ARTIFACTS"

In [ ]:
from IPython.display import Image, display
display(Image(f"{ARTIFACTS}/eda/class_distribution.png", width=700))
display(Image(f"{ARTIFACTS}/eda/samples.png", width=700))

## 4. (Optional) compare backbones
Quick 3-epoch comparison on frozen backbones. Takes ~30–40 min on a T4. Skip it to train with EfficientNetB0 directly.

In [ ]:
# !python benchmark.py --epochs 3 --output-dir "$ARTIFACTS"

## 5. Train
Phase 1 trains the head on a frozen backbone. Phase 2 fine-tunes the whole backbone (BatchNorm stays frozen). The best checkpoint is kept.

In [ ]:
!python train.py --backbone efficientnetb0 --mixed-precision --output-dir "$ARTIFACTS"

### (Optional) squeeze out the last few tenths of a percent
Continue fine-tuning from the best checkpoint at lower learning rates. This took the model from 99.19% to 99.39% test accuracy.

In [ ]:
!python train.py --init-from "$ARTIFACTS/model.keras" --output-dir "$ARTIFACTS/round2" --finetune-epochs 12 --finetune-lr 1e-5 --mixed-precision --tta
!python train.py --init-from "$ARTIFACTS/round2/model.keras" --output-dir "$ARTIFACTS/round3" --finetune-epochs 10 --finetune-lr 5e-6 --mixed-precision --tta
ARTIFACTS = f"{ARTIFACTS}/round3"

## 6. Results

In [ ]:
import json
print(json.dumps(json.load(open(f"{ARTIFACTS}/metrics.json")), indent=2, ensure_ascii=False))
display(Image(f"{ARTIFACTS}/training_curves.png", width=900))
display(Image(f"{ARTIFACTS}/confusion_matrix.png", width=900))
print(open(f"{ARTIFACTS}/classification_report.txt").read())

## 7. Predict on your own image

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    !python predict.py "{name}" --artifacts "$ARTIFACTS"

## 8. Web demo
Opens a public Gradio link you can try on your phone.

In [ ]:
!python app.py --artifacts "$ARTIFACTS" --share